# 중앙일보 기사 크롤링

In [27]:
# 필요 모듈 import
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException, NoSuchElementException
import csv, time
import pandas as pd

In [ ]:
# Chrome headless 설정
options = Options()
options.add_argument("--headless")
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

# 초기 페이지 접속
driver.get("https://www.joongang.co.kr/realestate?page=1")
wait = WebDriverWait(driver, 5)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#container > section > div.contents_bottom.float_left > section:nth-child(2) > nav > ul > li.page_next > a')))

MAX_PAGE = 1
article_links = []

# 기사 목록 수집
for i in range(MAX_PAGE):
    try:
        print(f"{i+1} 번째 페이지 수집 중...")
        time.sleep(2)
        a_tags = driver.find_elements(By.CSS_SELECTOR, '#story_list a')
        for a in a_tags:
            href = a.get_attribute('href')
            if href:
                article_links.append(href)

        next_page_btn = driver.find_element(By.CSS_SELECTOR, '#container > section > div.contents_bottom.float_left > section:nth-child(2) > nav > ul > li.page_next > a')
        next_page_btn.click()
    except (ElementClickInterceptedException, NoSuchElementException) as e:
        print(f"[예외 발생] 다음 페이지 없음: {e}")
        break

article_list = list(set(article_links))
print(f"\n총 {len(article_list)}개의 기사 링크를 수집했습니다.")

# 기사 내용 수집
article_data = {}

for i, url in enumerate(article_list):
    try:
        driver.get(url)
        print(f"{len(article_list)}개 중 {i+1}번째 기사 크롤링 중...")
        time.sleep(2)

        article_section = driver.find_element(By.CSS_SELECTOR, "#article_body")
        paragraphs = article_section.find_elements(By.TAG_NAME, 'p')
        full_text = ''.join([p.text.strip() for p in paragraphs if p.text.strip()])

        time_element = driver.find_element(By.CSS_SELECTOR, 'time[itemprop="datePublished"]')
        published_date = time_element.get_attribute('datetime')

        article_data[url] = {"text": full_text, "date": published_date}

    except Exception as e:
        print(f"{i+1}번째 기사 오류 발생: {e}")
        continue

driver.quit()

1 번째 페이지 수집 중...

총 24개의 기사 링크를 수집했습니다.
24개 중 1번째 기사 크롤링 중...
24개 중 2번째 기사 크롤링 중...
24개 중 3번째 기사 크롤링 중...
24개 중 4번째 기사 크롤링 중...
24개 중 5번째 기사 크롤링 중...
24개 중 6번째 기사 크롤링 중...
24개 중 7번째 기사 크롤링 중...
24개 중 8번째 기사 크롤링 중...
24개 중 9번째 기사 크롤링 중...
24개 중 10번째 기사 크롤링 중...
24개 중 11번째 기사 크롤링 중...
24개 중 12번째 기사 크롤링 중...
24개 중 13번째 기사 크롤링 중...
24개 중 14번째 기사 크롤링 중...
24개 중 15번째 기사 크롤링 중...
24개 중 16번째 기사 크롤링 중...
24개 중 17번째 기사 크롤링 중...
24개 중 18번째 기사 크롤링 중...
24개 중 19번째 기사 크롤링 중...
24개 중 20번째 기사 크롤링 중...
24개 중 21번째 기사 크롤링 중...
24개 중 22번째 기사 크롤링 중...
24개 중 23번째 기사 크롤링 중...
24개 중 24번째 기사 크롤링 중...
24번째 기사 오류 발생: Message: no such element: Unable to locate element: {"method":"css selector","selector":"time[itemprop="datePublished"]"}
  (Session info: chrome=137.0.7151.104); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
0   chromedriver                        0x00000001047603cc cxxbridge1$str$ptr + 28

In [26]:

# CSV 저장
with open('../data/raw/news/joongang_ilbo.csv', 'w', newline="", encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['URL', 'content', 'date'])
    for url, content in article_data.items():
        writer.writerow([url, content["text"], content["date"]])

print("크롤링 및 저장이 완료되었습니다.")

크롤링 및 저장이 완료되었습니다.


# 뉴스기사 전처리

## 결측치 처리 및 월별로 정렬

In [56]:
joongang = pd.read_csv('../data/raw/news/joongang_ilbo.csv')

In [57]:
joongang.dropna(inplace=True)

In [58]:
joongang.drop(["URL"], axis=1,inplace=True)

In [59]:
type(joongang['date'][1])

str

In [60]:
joongang['date'] = pd.to_datetime(joongang['date'].str[:10])

In [66]:
# 날짜순으로 정렬하기
joongang.sort_values(by='date', ascending=False, inplace=True)
# 컬럼 순서 변경
joongang = joongang[['date','content']]
joongang = joongang.reset_index(drop=True)

In [67]:
joongang.head()

,date,content
0,2025-06-16,서울 시내 아파트의 모습. 뉴스1서울 부동산 시장이 뜨겁게 달아오르고 있다. ‘지금...
1,2025-06-14,세종시의 한 부동산중개업소 매물판. 다시 매물이 늘고 있다. 황정일 기자“지금은 분...
2,2025-06-13,토지거래허가제(토허제) 적용을 받지 않는 서울 성동구 집값 상승세가 심상찮다. 12...
3,2025-06-13,서울 아파트 가격이 19주 연속 오르며 상승 폭을 키우고 있다. 12일 한국부동산원...
4,2025-06-12,서울 송파구 아파트의 모습. 연합뉴스서울의 전용 59㎡ 소형 아파트 분양가가 이른바...


In [68]:
joongang.to_csv('../data/interim/news/interim_joongang.csv', index=False)